In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Student Performance Prediction — Exploratory Data Analysis\n",
    "\n",
    "This notebook performs a thorough EDA on the student dataset to understand\n",
    "feature distributions, correlations, and class balance before modelling."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import os\n",
    "import sys\n",
    "\n",
    "# Add project root to path\n",
    "PROJECT_ROOT = os.path.abspath(os.path.join(\"..\"))\n",
    "sys.path.insert(0, PROJECT_ROOT)\n",
    "\n",
    "import pandas as pd\n",
    "import numpy as np\n",
    "import matplotlib.pyplot as plt\n",
    "import seaborn as sns\n",
    "\n",
    "sns.set_style(\"whitegrid\")\n",
    "sns.set_palette(\"husl\")\n",
    "\n",
    "print(\"Libraries loaded successfully.\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 1. Load the Dataset"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "DATA_PATH = os.path.join(PROJECT_ROOT, \"data\", \"student_data.csv\")\n",
    "df = pd.read_csv(DATA_PATH)\n",
    "\n",
    "print(f\"Shape: {df.shape}\")\n",
    "df.head(10)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 2. Dataset Inspection"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "print(\"--- Column Data Types ---\")\n",
    "print(df.dtypes)\n",
    "print()\n",
    "print(\"--- Dataset Info ---\")\n",
    "df.info()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 3. Missing Value Analysis"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "missing = df.isnull().sum()\n",
    "missing_pct = (missing / len(df) * 100).round(2)\n",
    "\n",
    "missing_df = pd.DataFrame({\n",
    "    \"Missing Values\": missing,\n",
    "    \"Percentage (%)\": missing_pct,\n",
    "})\n",
    "print(missing_df)\n",
    "print(f\"\\nTotal missing values: {missing.sum()}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 4. Duplicate Check"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "duplicates = df.duplicated(subset=\"student_id\").sum()\n",
    "print(f\"Duplicate student IDs: {duplicates}\")\n",
    "\n",
    "if duplicates > 0:\n",
    "    df.drop_duplicates(subset=\"student_id\", inplace=True)\n",
    "    print(f\"Duplicates removed. New shape: {df.shape}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 5. Statistical Summary"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "df.describe().round(2)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 6. Class Distribution Analysis"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "class_counts = df[\"final_status\"].value_counts()\n",
    "class_pct = (class_counts / len(df) * 100).round(2)\n",
    "\n",
    "print(\"--- Class Counts ---\")\n",
    "print(class_counts)\n",
    "print()\n",
    "print(\"--- Class Percentages ---\")\n",
    "print(class_pct)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "fig, axes = plt.subplots(1, 2, figsize=(12, 5))\n",
    "\n",
    "# Bar chart\n",
    "sns.countplot(data=df, x=\"final_status\", ax=axes[0], palette=[\"#e74c3c\", \"#2ecc71\"])\n",
    "axes[0].set_title(\"Class Distribution (Count)\", fontsize=14)\n",
    "axes[0].set_xlabel(\"Final Status\")\n",
    "axes[0].set_ylabel(\"Count\")\n",
    "for p in axes[0].patches:\n",
    "    axes[0].annotate(\n",
    "        f\"{int(p.get_height())}\",\n",
    "        (p.get_x() + p.get_width() / 2, p.get_height()),\n",
    "        ha=\"center\", va=\"bottom\", fontsize=12, fontweight=\"bold\",\n",
    "    )\n",
    "\n",
    "# Pie chart\n",
    "axes[1].pie(\n",
    "    class_counts.values,\n",
    "    labels=class_counts.index,\n",
    "    autopct=\"%1.1f%%\",\n",
    "    colors=[\"#e74c3c\", \"#2ecc71\"],\n",
    "    startangle=90,\n",
    "    textprops={\"fontsize\": 12},\n",
    "    explode=[0.03, 0.03],\n",
    ")\n",
    "axes[1].set_title(\"Class Distribution (Percentage)\", fontsize=14)\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.savefig(os.path.join(PROJECT_ROOT, \"reports\", \"class_distribution.png\"), dpi=150)\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 7. Feature Distribution Analysis"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "FEATURE_COLS = [\n",
    "    \"hours_studied\",\n",
    "    \"attendance_percentage\",\n",
    "    \"previous_marks\",\n",
    "    \"assignments_completed\",\n",
    "    \"class_tests_score\",\n",
    "    \"participation_score\",\n",
    "]\n",
    "\n",
    "fig, axes = plt.subplots(2, 3, figsize=(16, 10))\n",
    "axes = axes.flatten()\n",
    "\n",
    "for i, col in enumerate(FEATURE_COLS):\n",
    "    sns.histplot(data=df, x=col, kde=True, ax=axes[i], hue=\"final_status\",\n",
    "                 palette={\"Fail\": \"#e74c3c\", \"Pass\": \"#2ecc71\"}, alpha=0.6)\n",
    "    axes[i].set_title(f\"Distribution of {col.replace('_', ' ').title()}\", fontsize=12)\n",
    "    axes[i].set_xlabel(\"\")\n",
    "\n",
    "plt.suptitle(\"Feature Distributions by Class\", fontsize=16, y=1.02)\n",
    "plt.tight_layout()\n",
    "plt.savefig(os.path.join(PROJECT_ROOT, \"reports\", \"feature_distributions.png\"), dpi=150, bbox_inches=\"tight\")\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 8. Box Plot Analysis"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "fig, axes = plt.subplots(2, 3, figsize=(16, 10))\n",
    "axes = axes.flatten()\n",
    "\n",
    "for i, col in enumerate(FEATURE_COLS):\n",
    "    sns.boxplot(data=df, x=\"final_status\", y=col, ax=axes[i],\n",
    "                palette={\"Fail\": \"#e74c3c\", \"Pass\": \"#2ecc71\"})\n",
    "    axes[i].set_title(f\"{col.replace('_', ' ').title()} by Status\", fontsize=12)\n",
    "    axes[i].set_xlabel(\"\")\n",
    "\n",
    "plt.suptitle(\"Box Plots — Feature vs Final Status\", fontsize=16, y=1.02)\n",
    "plt.tight_layout()\n",
    "plt.savefig(os.path.join(PROJECT_ROOT, \"reports\", \"boxplots.png\"), dpi=150, bbox_inches=\"tight\")\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 9. Correlation Analysis"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Encode target for correlation\n",
    "df_encoded = df.copy()\n",
    "df_encoded[\"final_status_encoded\"] = df_encoded[\"final_status\"].map({\"Fail\": 0, \"Pass\": 1})\n",
    "\n",
    "corr_cols = FEATURE_COLS + [\"final_status_encoded\"]\n",
    "corr_matrix = df_encoded[corr_cols].corr().round(3)\n",
    "\n",
    "print(\"--- Correlation Matrix ---\")\n",
    "print(corr_matrix)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "plt.figure(figsize=(10, 8))\n",
    "mask = np.triu(np.ones_like(corr_matrix, dtype=bool))\n",
    "\n",
    "sns.heatmap(\n",
    "    corr_matrix,\n",
    "    annot=True,\n",
    "    fmt=\".2f\",\n",
    "    cmap=\"RdBu_r\",\n",
    "    center=0,\n",
    "    mask=mask,\n",
    "    square=True,\n",
    "    linewidths=0.5,\n",
    "    xticklabels=[c.replace(\"_\", \" \").title() for c in corr_cols],\n",
    "    yticklabels=[c.replace(\"_\", \" \").title() for c in corr_cols],\n",
    ")\n",
    "plt.title(\"Feature Correlation Heatmap\", fontsize=16, pad=20)\n",
    "plt.xticks(rotation=45, ha=\"right\")\n",
    "plt.tight_layout()\n",
    "plt.savefig(os.path.join(PROJECT_ROOT, \"reports\", \"correlation_heatmap.png\"), dpi=150)\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 10. Correlation with Target"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "target_corr = corr_matrix[\"final_status_encoded\"].drop(\"final_status_encoded\").sort_values(ascending=False)\n",
    "\n",
    "plt.figure(figsize=(8, 5))\n",
    "colors = [\"#2ecc71\" if v > 0 else \"#e74c3c\" for v in target_corr.values]\n",
    "sns.barplot(x=target_corr.values, y=target_corr.index, palette=colors)\n",
    "plt.title(\"Feature Correlation with Final Status (Pass=1)\", fontsize=14)\n",
    "plt.xlabel(\"Correlation Coefficient\")\n",
    "plt.ylabel(\"\")\n",
    "plt.xlim(-0.1, 1.0)\n",
    "\n",
    "for i, v in enumerate(target_corr.values):\n",
    "    plt.text(v + 0.01, i, f\"{v:.3f}\", va=\"center\", fontsize=11, fontweight=\"bold\")\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.savefig(os.path.join(PROJECT_ROOT, \"reports\", \"target_correlation.png\"), dpi=150)\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 11. Pair Plot (Selected Features)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Select top 4 features by correlation with target for pair plot\n",
    "top_features = target_corr.abs().sort_values(ascending=False).head(4).index.tolist()\n",
    "pairplot_cols = top_features + [\"final_status\"]\n",
    "\n",
    "sns.pairplot(\n",
    "    df[pairplot_cols],\n",
    "    hue=\"final_status\",\n",
    "    palette={\"Fail\": \"#e74c3c\", \"Pass\": \"#2ecc71\"},\n",
    "    diag_kind=\"kde\",\n",
    "    plot_kws={\"alpha\": 0.4, \"s\": 30},\n",
    ")\n",
    "plt.suptitle(\"Pair Plot — Top 4 Features\", fontsize=16, y=1.02)\n",
    "plt.savefig(os.path.join(PROJECT_ROOT, \"reports\", \"pairplot.png\"), dpi=150, bbox_inches=\"tight\")\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 12. Key Findings Summary\n",
    "\n",
    "| Finding | Detail |\n",
    "|---------|--------|\n",
    "| Dataset size | 1000 records, 8 columns |\n",
    "| Missing values | None |\n",
    "| Duplicates | None |\n",
    "| Class balance | ~60% Pass, ~40% Fail (slight imbalance) |\n",
    "| Most predictive features | Previous marks, class tests score, attendance percentage |\n",
    "| Feature correlations | Moderate positive correlations among features; all positively correlated with Pass |\n",
    "| Outliers | Some low-end outliers in hours_studied and participation_score |\n",
    "\n",
    "### Recommendations for Modelling\n",
    "1. No imputation needed — no missing values.\n",
    "2. Standard scaling recommended due to different feature scales.\n",
    "3. All 6 features show meaningful correlation with the target — keep all.\n",
    "4. Consider stratified train/test split to preserve class balance."
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python",
   "version": "3.11.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}